In [22]:
import os
import h5py
import numpy as np
import pandas as pd

# ==== Folder containing all .h5 files ====
folder_path = "C:/Users/umair.muhammad/Documents/PhD/Research Work/FedLearn/training/All_Nome/las_csvs/labeled_h5"  # change to your folder path

# ==== Find all HDF5 files ====
h5_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(".h5")]

print(f"📂 Found {len(h5_files)} HDF5 files:\n")
for f in h5_files:
    print(" -", os.path.basename(f))
print("\n")

# ==== Combine all data ====
all_points = []
all_labels = []
summary = []  # <-- store per-site stats

for file_path in h5_files:
    try:
        with h5py.File(file_path, "r") as f:
            data = np.array(f["data"])
            label = np.array(f["label"]).flatten()  # ensure 1D

            all_points.append(data)
            all_labels.append(label)

            total_points = len(label)
            damaged_points = np.sum(label == 1)
            nondamaged_points = np.sum(label == 0)
            damage_ratio = 100 * damaged_points / total_points if total_points > 0 else 0

            summary.append({
                "Site": os.path.basename(file_path),
                "Total Points": total_points,
                "Damaged Points": damaged_points,
                "Non-Damaged Points": nondamaged_points,
                "Damage Ratio (%)": round(damage_ratio, 3)
            })

            print(f"✅ Loaded: {os.path.basename(file_path)} | Total: {total_points} | Damaged: {damaged_points}")

    except Exception as e:
        print(f"❌ Error loading {file_path}: {e}")

# ==== Concatenate all ====
if all_points:
    combined_points = np.vstack(all_points)
    combined_labels = np.hstack(all_labels)

    print("\n🎯 Combined data loaded successfully.")
    print("Total points shape:", combined_points.shape)
    print("Total labels shape:", combined_labels.shape)
    print("Unique labels:", np.unique(combined_labels, return_counts=True))

    # ==== Create summary DataFrame ====
    df_summary = pd.DataFrame(summary)
    print("\n📊 Damage summary per site:")
    print(df_summary)

    # ==== Save summary to CSV ====
    out_csv = os.path.join(folder_path, "damage_summary.csv")
    df_summary.to_csv(out_csv, index=False)
    print(f"\n💾 Saved summary CSV to: {out_csv}")

else:
    print("⚠️ No valid HDF5 files loaded.")


📂 Found 4 HDF5 files:

 - Site1.h5
 - Site2.h5
 - Site3.h5
 - Site5.h5


✅ Loaded: Site1.h5 | Total: 439928 | Damaged: 29308
✅ Loaded: Site2.h5 | Total: 860023 | Damaged: 10461
✅ Loaded: Site3.h5 | Total: 1076806 | Damaged: 442729
✅ Loaded: Site5.h5 | Total: 378119 | Damaged: 29962

🎯 Combined data loaded successfully.
Total points shape: (2754876, 3)
Total labels shape: (2754876,)
Unique labels: (array([0, 1]), array([2242416,  512460], dtype=int64))

📊 Damage summary per site:
       Site  Total Points  Damaged Points  Non-Damaged Points  \
0  Site1.h5        439928           29308              410620   
1  Site2.h5        860023           10461              849562   
2  Site3.h5       1076806          442729              634077   
3  Site5.h5        378119           29962              348157   

   Damage Ratio (%)  
0             6.662  
1             1.216  
2            41.115  
3             7.924  

💾 Saved summary CSV to: C:/Users/umair.muhammad/Documents/PhD/Research Work/Fed

In [23]:
#test train Split
import os
import h5py
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# ==== Folder containing all .h5 files ====
folder_path = "C:/Users/umair.muhammad/Documents/PhD/Research Work/FedLearn/training/All_Nome/las_csvs/labeled_h5"  # change this to your folder

# ==== Collect all HDF5 files ====
h5_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(".h5")]

print(f"📂 Found {len(h5_files)} HDF5 files.\n")

all_points, all_labels = [], []
summary = []

# ==== Load and combine all point clouds ====
for file_path in h5_files:
    try:
        with h5py.File(file_path, "r") as f:
            data = np.array(f["data"])
            label = np.array(f["label"]).flatten()
            all_points.append(data)
            all_labels.append(label)

            total_points = len(label)
            damaged = np.sum(label == 1)
            undamaged = np.sum(label == 0)
            ratio = 100 * damaged / total_points

            summary.append({
                "File": os.path.basename(file_path),
                "Total Points": total_points,
                "Damaged Points": damaged,
                "Undamaged Points": undamaged,
                "Damage Ratio (%)": round(ratio, 3)
            })

            print(f"✅ Loaded {os.path.basename(file_path)} | Points: {total_points} | Damaged: {damaged}")
    except Exception as e:
        print(f"❌ Error loading {file_path}: {e}")

# ==== Combine all files ====
combined_points = np.vstack(all_points)
combined_labels = np.hstack(all_labels)

print("\n🎯 Combined dataset summary:")
print("Total points:", combined_points.shape[0])
print("Unique labels:", np.unique(combined_labels, return_counts=True))

# ==== Create patches for PointNet ====
num_points_per_patch = 1024
num_total_points = combined_points.shape[0]
num_patches = num_total_points // num_points_per_patch
has_partial_patch = (num_total_points % num_points_per_patch != 0)
if has_partial_patch:
    num_patches += 1

print(f"\nPreparing patches: {num_patches} (each {num_points_per_patch} points)")

point_patches = []
label_patches = []

for i in range(num_patches):
    start = i * num_points_per_patch
    end = min((i + 1) * num_points_per_patch, num_total_points)
    pts = combined_points[start:end]
    lbl = combined_labels[start:end]

    if has_partial_patch and i == num_patches - 1 and pts.shape[0] < num_points_per_patch:
        pad = num_points_per_patch - pts.shape[0]
        pts = np.pad(pts, ((0, pad), (0, 0)), mode='constant', constant_values=0)
        lbl = np.pad(lbl, (0, pad), mode='constant', constant_values=0)

    point_patches.append(pts)
    label_patches.append(lbl)

point_patches = np.array(point_patches)
label_patches = np.array(label_patches)

print("\n✅ Patch generation complete.")
print("point_patches:", point_patches.shape)
print("label_patches:", label_patches.shape)

# ==== Split 80/20 for training and testing ====
train_points, test_points, train_labels, test_labels = train_test_split(
    point_patches, label_patches, test_size=0.2, random_state=42, shuffle=True
)

print("\n📊 Split summary:")
print(f"Training patches: {len(train_points)}")
print(f"Testing patches: {len(test_points)}")

# ==== Save to .npz for later model input ====
save_path = os.path.join(folder_path, "road_damage_patches.npz")
np.savez(save_path, 
         train_points=train_points, 
         train_labels=train_labels, 
         test_points=test_points, 
         test_labels=test_labels)

print(f"\n💾 Saved processed dataset to: {save_path}")


📂 Found 4 HDF5 files.

✅ Loaded Site1.h5 | Points: 439928 | Damaged: 29308
✅ Loaded Site2.h5 | Points: 860023 | Damaged: 10461
✅ Loaded Site3.h5 | Points: 1076806 | Damaged: 442729
✅ Loaded Site5.h5 | Points: 378119 | Damaged: 29962

🎯 Combined dataset summary:
Total points: 2754876
Unique labels: (array([0, 1]), array([2242416,  512460], dtype=int64))

Preparing patches: 2691 (each 1024 points)

✅ Patch generation complete.
point_patches: (2691, 1024, 3)
label_patches: (2691, 1024)

📊 Split summary:
Training patches: 2152
Testing patches: 539

💾 Saved processed dataset to: C:/Users/umair.muhammad/Documents/PhD/Research Work/FedLearn/training/All_Nome/las_csvs/labeled_h5\road_damage_patches.npz


In [ ]:
# # Overview

# Data preparation

# Model definition / adaptation

# Loss & optimizer setup (normal vs class‑weighted)

# Training

# Evaluation (per‑class IoU, F1)



In [24]:
import numpy as np
from sklearn.model_selection import train_test_split

# Assume point_patches: shape (N, num_points, 3)
#        label_patches: shape (N, num_points) with integer labels 0/1

num_points = point_patches.shape[1]

# Split into train/test
Train_X_Data, Test_X_Data, Train_Y_Data, Test_Y_Data = train_test_split(
    point_patches, label_patches, test_size=0.20, random_state=42, shuffle=True
)

print("Train_X_Data:", Train_X_Data.shape)
print("Train_Y_Data:", Train_Y_Data.shape)
print("Test_X_Data:", Test_X_Data.shape)
print("Test_Y_Data:", Test_Y_Data.shape)


Train_X_Data: (2152, 1024, 3)
Train_Y_Data: (2152, 1024)
Test_X_Data: (539, 1024, 3)
Test_Y_Data: (539, 1024)


In [25]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# -----------------------------
# Utility Blocks
# -----------------------------
def mlp_block(x, layer_sizes, name_prefix="mlp"):
    for i, size in enumerate(layer_sizes):
        x = layers.Conv1D(size, 1, activation='relu', name=f"{name_prefix}_conv_{i}")(x)
        x = layers.BatchNormalization(name=f"{name_prefix}_bn_{i}")(x)
    return x


# -----------------------------
# Set Abstraction Module (SA)
# -----------------------------
def pointnet_sa_module(x, npoint, radius, nsample, mlp, group_all, name):
    # In real PointNet++, this does sampling & grouping. Here simplified.
    for i, out_size in enumerate(mlp):
        x = layers.Conv1D(out_size, 1, activation='relu', name=f"{name}_conv_{i}")(x)
        x = layers.BatchNormalization(name=f"{name}_bn_{i}")(x)
    return x


# -----------------------------
# Feature Propagation Module (FP)
# -----------------------------
def pointnet_fp_module(skip, x, mlp, name):
    # Upsample x to match skip's number of points
    def upsample_to_skip(tensors):
        x_in, skip_in = tensors
        num_skip = tf.shape(skip_in)[1]
        x_in = tf.expand_dims(x_in, axis=2)  # (B, N, 1, C)
        x_in = tf.image.resize(x_in, [num_skip, 1])  # linear interpolation
        x_in = tf.squeeze(x_in, axis=2)  # back to (B, N, C)
        return x_in

    x = layers.Lambda(upsample_to_skip, name=f"{name}_upsample")([x, skip])
    x = layers.Concatenate(name=f"{name}_concat")([x, skip])
    x = mlp_block(x, mlp, name_prefix=name)
    return x


# -----------------------------
# PointNet++ Semantic Segmentation Model
# -----------------------------
def create_pointnet2_semseg(num_points=1024, num_classes=2):
    inputs = layers.Input(shape=(num_points, 3))

    # Encoder: Set Abstraction layers
    sa1 = pointnet_sa_module(inputs, npoint=512, radius=0.1, nsample=32, mlp=[64, 64, 128],
                             group_all=False, name="sa1")
    sa2 = pointnet_sa_module(sa1, npoint=128, radius=0.2, nsample=64, mlp=[128, 128, 256],
                             group_all=False, name="sa2")
    sa3 = pointnet_sa_module(sa2, npoint=32, radius=0.4, nsample=128, mlp=[256, 256, 512],
                             group_all=False, name="sa3")
    sa4 = pointnet_sa_module(sa3, npoint=None, radius=None, nsample=None, mlp=[512, 512, 1024],
                             group_all=True, name="sa4")

    # Decoder: Feature Propagation layers
    fp3 = pointnet_fp_module(sa3, sa4, mlp=[512, 512], name="fp3")
    fp2 = pointnet_fp_module(sa2, fp3, mlp=[512, 256], name="fp2")
    fp1 = pointnet_fp_module(sa1, fp2, mlp=[256, 128], name="fp1")

    # NEW: upsample back to full resolution (1024)
    fp0 = pointnet_fp_module(inputs, fp1, mlp=[128, 128, 128], name="fp0")

    # Output layer
    x = layers.Conv1D(num_classes, 1, activation='softmax', name='segmentation_head')(fp0)
    model = Model(inputs=inputs, outputs=x, name="PointNet2_Semantic_Segmentation")

    # Compile model
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


# -----------------------------
# Create Model and Train
# -----------------------------
num_points = 1024
num_classes = 2

model = create_pointnet2_semseg(num_points=num_points, num_classes=num_classes)
model.summary()

# Example (use your actual training/test data)
# Train_X_Data: (N, 1024, 3)
# Train_Y_Data: (N, 1024)
# Test_X_Data: (M, 1024, 3)
# Test_Y_Data: (M, 1024)

history = model.fit(
    Train_X_Data, Train_Y_Data,
    validation_data=(Test_X_Data, Test_Y_Data),
    epochs=5,
    batch_size=16,
    shuffle=True
)


Model: "PointNet2_Semantic_Segmentation"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_4 (InputLayer)           [(None, 1024, 3)]    0           []                               
                                                                                                  
 sa1_conv_0 (Conv1D)            (None, 1024, 64)     256         ['input_4[0][0]']                
                                                                                                  
 sa1_bn_0 (BatchNormalization)  (None, 1024, 64)     256         ['sa1_conv_0[0][0]']             
                                                                                                  
 sa1_conv_1 (Conv1D)            (None, 1024, 64)     4160        ['sa1_bn_0[0][0]']               
                                                                    

In [ ]:
# Loss & Optimizer Setup
# Normal Loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)


In [11]:
history = model.fit(
    Train_X_Data, Train_Y_Data,
    validation_data=(Test_X_Data, Test_Y_Data),
    epochs=5,            # adjust
    batch_size=16,        # adjust depending on memory
    shuffle=True
)


1112/1112 [==============================] - 1208s 1s/step - loss: 0.1373 - accuracy: 0.9663 - val_loss: 74.4158 - val_accuracy: 0.9654
Epoch 2/5
1112/1112 [==============================] - 1215s 1s/step - loss: 0.1367 - accuracy: 0.9663 - val_loss: 15.2594 - val_accuracy: 0.9654
Epoch 3/5
 803/1112 [====================>.........] - ETA: 5:22 - loss: 0.1365 - accuracy: 0.9664

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [26]:
from sklearn.metrics import f1_score
import numpy as np

y_pred_probs = model.predict(Test_X_Data, batch_size=16)
y_pred_labels = np.argmax(y_pred_probs, axis=-1)

y_true_flat = Test_Y_Data.flatten()
y_pred_flat = y_pred_labels.flatten()

classes = np.unique(y_true_flat)
print("=== Per‑class metrics ===")
for c in classes:
    mask = (y_true_flat == c)
    class_acc = np.sum(y_pred_flat[mask] == y_true_flat[mask]) / np.sum(mask)
    y_true_binary = (y_true_flat == c).astype(int)
    y_pred_binary = (y_pred_flat == c).astype(int)
    f1 = f1_score(y_true_binary, y_pred_binary)
    intersection = np.sum(y_true_binary & y_pred_binary)
    union = np.sum(y_true_binary | y_pred_binary)
    iou = intersection / union
    print(f"Class {c}: Accuracy={class_acc:.4f}, F1={f1:.4f}, IoU={iou:.4f}")

overall_acc = np.mean(y_pred_flat == y_true_flat)
print(f"\nOverall point‐wise accuracy: {overall_acc:.4f}")


34/34 [==============================] - 5s 124ms/step
=== Per‑class metrics ===
Class 0: Accuracy=0.0000, F1=0.0000, IoU=0.0000
Class 1: Accuracy=1.0000, F1=0.3271, IoU=0.1955

Overall point‐wise accuracy: 0.1955


In [27]:
# Class‑Weighted Loss
import tensorflow as tf

class_weights= {0: 0.5160885637599996, 1: 16.03898804948429}

def weighted_sparse_categorical_crossentropy(class_weights):
    def loss(y_true, y_pred):
        # y_true: (batch_size, num_points)
        # y_pred: (batch_size, num_points, num_classes)
        y_true_int = tf.cast(y_true, tf.int32)
        weights = tf.gather(tf.constant([class_weights[0], class_weights[1]], dtype=tf.float32),
                            y_true_int)
        scce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
        weighted = scce * weights
        return tf.reduce_mean(weighted)
    return loss

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=weighted_sparse_categorical_crossentropy(class_weights),
    metrics=["accuracy"]
)


In [16]:
history = model.fit(
    Train_X_Data, Train_Y_Data,
    validation_data=(Test_X_Data, Test_Y_Data),
    epochs=5,            # adjust
    batch_size=16,        # adjust depending on memory
    shuffle=True
)


Epoch 1/5
1112/1112 [==============================] - 1268s 1s/step - loss: 0.5975 - accuracy: 0.5832 - val_loss: 74532.1797 - val_accuracy: 0.9654
Epoch 2/5
1112/1112 [==============================] - 1232s 1s/step - loss: 0.5736 - accuracy: 0.5708 - val_loss: 275.6856 - val_accuracy: 0.9654
Epoch 3/5
1112/1112 [==============================] - 1219s 1s/step - loss: 0.5739 - accuracy: 0.5719 - val_loss: 366.0425 - val_accuracy: 0.9654
Epoch 4/5
1112/1112 [==============================] - 1212s 1s/step - loss: 0.5707 - accuracy: 0.5776 - val_loss: 4452.6904 - val_accuracy: 0.9654
Epoch 5/5
1112/1112 [==============================] - 1263s 1s/step - loss: 0.5629 - accuracy: 0.5786 - val_loss: 2205309.7500 - val_accuracy: 0.9654


In [ ]:
from sklearn.metrics import f1_score
import numpy as np

y_pred_probs = model.predict(Test_X_Data, batch_size=16)
y_pred_labels = np.argmax(y_pred_probs, axis=-1)

y_true_flat = Test_Y_Data.flatten()
y_pred_flat = y_pred_labels.flatten()

classes = np.unique(y_true_flat)
print("=== Per‑class metrics ===")
for c in classes:
    mask = (y_true_flat == c)
    class_acc = np.sum(y_pred_flat[mask] == y_true_flat[mask]) / np.sum(mask)
    y_true_binary = (y_true_flat == c).astype(int)
    y_pred_binary = (y_pred_flat == c).astype(int)
    f1 = f1_score(y_true_binary, y_pred_binary)
    intersection = np.sum(y_true_binary & y_pred_binary)
    union = np.sum(y_true_binary | y_pred_binary)
    iou = intersection / union
    print(f"Class {c}: Accuracy={class_acc:.4f}, F1={f1:.4f}, IoU={iou:.4f}")

overall_acc = np.mean(y_pred_flat == y_true_flat)
print(f"\nOverall point‐wise accuracy: {overall_acc:.4f}")


In [17]:
# PointNet++ Semantic Segmentation Pipeline
# Compatible with: point_patches (N, 1024, 3), label_patches (N, 1024)

import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

# ===============================
# Step 1: Data Preparation
# ===============================
num_points = 1024
num_classes = 2

# If your patches are larger, we randomly downsample to 1024
def sample_points(points, n_points=1024):
    if points.shape[0] > n_points:
        idx = np.random.choice(points.shape[0], n_points, replace=False)
        return points[idx]
    return points

# Downsample all patches
point_patches_1024 = np.array([sample_points(p, num_points) for p in point_patches])
label_patches_1024 = np.array([sample_points(l, num_points) for l in label_patches])

# Train/test split
Train_X_Data, Test_X_Data, Train_Y_Data, Test_Y_Data = train_test_split(
    point_patches_1024, label_patches_1024, test_size=0.20, random_state=42, shuffle=True
)

print("Train_X_Data:", Train_X_Data.shape)
print("Train_Y_Data:", Train_Y_Data.shape)
print("Test_X_Data:", Test_X_Data.shape)
print("Test_Y_Data:", Test_Y_Data.shape)

# ===============================
# Step 2: Define PointNet++ Model
# ===============================
def pointnet_sa_module(inputs, npoint, radius, nsample, mlp, group_all=False):
    """
    Simplified Set Abstraction (SA) module
    """
    # This is a simplified version; for real PointNet++ use official set abstraction functions
    x = layers.Conv1D(mlp[0], 1, activation='relu')(inputs)
    x = layers.Conv1D(mlp[1], 1, activation='relu')(x)
    return x

def pointnet_fp_module(x, skip, mlp):
    """
    Simplified Feature Propagation module
    """
    x = layers.Concatenate()([x, skip])
    x = layers.Conv1D(mlp[0], 1, activation='relu')(x)
    x = layers.Conv1D(mlp[1], 1, activation='relu')(x)
    return x

def create_pointnet2_semseg(num_points=1024, num_classes=2):
    inputs = tf.keras.Input(shape=(num_points,3))
    # Set Abstraction Layers
    sa1 = pointnet_sa_module(inputs, npoint=512, radius=0.1, nsample=32, mlp=[64,64])
    sa2 = pointnet_sa_module(sa1, npoint=128, radius=0.2, nsample=64, mlp=[128,128])
    sa3 = pointnet_sa_module(sa2, npoint=None, radius=None, nsample=None, mlp=[256,512], group_all=True)
    
    # Feature Propagation
    fp2 = pointnet_fp_module(sa2, sa3, mlp=[256,256])
    fp1 = pointnet_fp_module(sa1, fp2, mlp=[256,128])
    fp0 = pointnet_fp_module(inputs, fp1, mlp=[128,128])
    
    outputs = layers.Conv1D(num_classes, 1, activation='softmax')(fp0)
    
    model = Model(inputs=inputs, outputs=outputs, name="PointNet2_SemSeg_1024")
    return model

model = create_pointnet2_semseg(num_points=num_points, num_classes=num_classes)
model.summary()

# ===============================
# Step 3a: Normal Loss
# ===============================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# ===============================
# Step 3b: Weighted Loss
# ===============================
class_weights = {0: 0.5160885637599996, 1: 16.03898804948429}

def weighted_sparse_categorical_crossentropy(class_weights):
    def loss_fn(y_true, y_pred):
        y_true_int = tf.cast(y_true, tf.int32)
        weights = tf.gather(tf.constant([class_weights[0], class_weights[1]], dtype=tf.float32), y_true_int)
        scce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
        weighted_loss = scce * weights
        return tf.reduce_mean(weighted_loss)
    return loss_fn

# To use weighted loss:
# model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
#               loss=weighted_sparse_categorical_crossentropy(class_weights),
#               metrics=['accuracy'])

# ===============================
# Step 4: Training
# ===============================
history = model.fit(
    Train_X_Data, Train_Y_Data,
    validation_data=(Test_X_Data, Test_Y_Data),
    epochs=50,
    batch_size=16,
    shuffle=True
)

# ===============================
# Step 5: Evaluation
# ===============================
y_pred_probs = model.predict(Test_X_Data, batch_size=16)
y_pred_labels = np.argmax(y_pred_probs, axis=-1)

y_true_flat = Test_Y_Data.flatten()
y_pred_flat = y_pred_labels.flatten()

classes = np.unique(y_true_flat)
print("=== Per-class metrics ===")
for c in classes:
    mask = (y_true_flat == c)
    class_acc = np.sum(y_pred_flat[mask] == y_true_flat[mask]) / np.sum(mask)
    y_true_binary = (y_true_flat == c).astype(int)
    y_pred_binary = (y_pred_flat == c).astype(int)
    f1 = f1_score(y_true_binary, y_pred_binary)
    intersection = np.sum((y_true_binary & y_pred_binary))
    union = np.sum((y_true_binary | y_pred_binary))
    iou = intersection / union
    print(f"Class {c}: Accuracy={class_acc:.4f}, F1={f1:.4f}, IoU={iou:.4f}")

overall_acc = np.mean(y_pred_flat == y_true_flat)
print(f"\nOverall point-wise accuracy: {overall_acc:.4f}")


Train_X_Data: (17792, 1024, 3)
Train_Y_Data: (17792, 1024)
Test_X_Data: (4448, 1024, 3)
Test_Y_Data: (4448, 1024)
Model: "PointNet2_SemSeg_1024"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 1024, 3)]    0           []                               
                                                                                                  
 conv1d_21 (Conv1D)             (None, 1024, 64)     256         ['input_3[0][0]']                
                                                                                                  
 conv1d_22 (Conv1D)             (None, 1024, 64)     4160        ['conv1d_21[0][0]']              
                                                                                                  
 conv1d_23 (Conv1D)             (None, 1024, 128)    8320      

KeyboardInterrupt: 